# Using input field from MMA simulations and testing TDSE parameters for this field

Our Pythonic interface access exactly the same solver from the same source. We can use it to get further insight into the microscopic physics or to verify the parameters of the simulation, check convergences...

This tutorial loads the data from a multiscale simulation and will get a closer look on the absorbing boundaries. As we need the source data, we use the ouput from the MMA-basics tutorial. Please complete this tutorial (workspace referred in `teach-me-mma`) or evaluate [the respective notebook](../mma_basics/teach_me_mma.ipynb) before this tutorial. It is expected that this tutorial is run with no absorber option and we will experiment with the absorber here.

First, we load standard modules and set the Pythonic CTDSE environment.

In [ ]:
## python modules used within this notebook
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation
import matplotlib.colors as colors
import os
import h5py
import sys
import mynumerics as mn
import units
from IPython.display import display, Markdown
from IPython.display import HTML

import HHG
import MMA_administration as MMA

matplotlib.rcParams['animation.embed_limit'] = 200.

# TDSE environment
from PythonTDSE import *

# Compiled dynamic C library
path_to_DLL = os.path.join(os.environ['TDSE_1D_BUILD'],'libsingleTDSE.so')
DLL = TDSE_DLL(path_to_DLL)

## Load data

Now we load th data from the hdf5-archive. First, we specify the path to the file (`trach-me-mma` tutorial). By modifying the path, any results can be accessed. Then there are two options `CTDSE` or `CUPRAD`. The fields are read from outputs, it means that both codes were already run. To experiment with CTDSE to adjust its parameters, this notebook can be used with the `CUPRAD` option. Next, there are selected points from the macroscopic grids by their indices. Note that these grids are not the same. *It might be helpful to open the hdf5 archive from jupyter to inspect the data instantly.* Next part of the code reads the data. Since the Pythonic TDSE provides the interface directly to the solver, we need to obtain the input parameters from the input file, it shows how to access them through [the administration module](../../shared_python/MMA_administration.py). Finally, these parameters are aligned into a dictionary to provide them to TDSE.

In [ ]:
results_path = os.path.join(os.environ['MULTISCALE_WORK_DIR'],'mma_basics')
filename     = 'results.h5'

field_source = 'CTDSE' # CTDSE or CUPRAD

z_select_index = 0
r_select_index = 0

with h5py.File(os.path.join(results_path,filename),'r') as f:
    # retrieve the field eithr from CUPRAD or CTDSE
    if  (field_source=='CTDSE'):
        Efield_path = MMA.paths['CTDSE_outputs']+'/Efield' 
        E = f[Efield_path][z_select_index,r_select_index,:]
        tgrid_path  = MMA.paths['CTDSE_outputs']+'/tgrid' 
    elif (field_source=='CUPRAD'):
        Efield_path = MMA.paths['CUPRAD_outputs']+'/output_field' 
        E = f[Efield_path][z_select_index,:,r_select_index]
        tgrid_path  = MMA.paths['CUPRAD_outputs']+'/tgrid' 
    else:
        raise NameError('only CTDSE or CUPRAD can be sources of the field')
    tgrid = f[tgrid_path][:]

    # obtain default gas and set CTDSE potential accordingly
    try: gas_preset = f[MMA.paths['global_inputs']+'/gas_preset'][()].decode()
    except: raise ValueError('This tutorial works with preset gas options')
    Eguess = -HHG.Ip_list[gas_preset]
    trg_a  = HHG.soft_Coulomb_a[gas_preset]

    # list of numerical parameters
    dt    = f[MMA.paths['CTDSE_inputs']+'/dt'][()]
    dx    = f[MMA.paths['CTDSE_inputs']+'/dx'][()]
    num_r = 2*f[MMA.paths['CTDSE_inputs']+'/Nx_max'][()]
    CV    = f[MMA.paths['CTDSE_inputs']+'/CV_criterion_of_GS'][()]

    # rangoe for the volumetric integration representing ionisation
    x_int = f[MMA.paths['CTDSE_inputs']+'/x_int'][()]
    

    # get absorber parameters
    absorber = {'type' : 0} # no absorber by default
    if MMA.paths['CTDSE_inputs']+'/absorber_type' in f: # absorber is optional in MMA pipeline
        absorber_type = f[MMA.paths['CTDSE_inputs']+'/absorber_type'][()]
        absorber['type'] = absorber_type
        if (absorber_type == 1):
            absorber['x_cap'] =  f[MMA.paths['CTDSE_inputs']+'/absorber_x_cap'][()]
            absorber['alpha'] =  f[MMA.paths['CTDSE_inputs']+'/absorber_alpha'][()]
        elif (absorber_type == 2):
            absorber['x_cap'] =  f[MMA.paths['CTDSE_inputs']+'/absorber_x_cap'][()]
            
# create the dictionary with the inputs
inputs_dict={'trg_a'    : trg_a,
             'Eguess'   : Eguess,
             'dt'       : dt,
             'dx'       : dx,
             'num_r'    : num_r, # 16000, #num_r,
             'CV'       : CV,
             'x_int'    : x_int,
             'absorber' : absorber, # {'type'  : 0}, # absorber,
             'writewft' : 1,
             'tprint'   : 1}

## Run TDSE

The inputs are prepared, we need just to initialise the inputs fo the solver and run it.

In [ ]:
inputs = inputs_def()
inputs.init_default_inputs(**inputs_dict)         # create the C-types input for the C-library
inputs.init_time_and_field(DLL, E = E, t = tgrid) # set our electric field as the input
DLL.init_GS(inputs)                               # initialise the ground state
output = outputs_def()                            # prepare the structure that holds the TDSE outputs 
DLL.call1DTDSE(inputs, output)                    # run TDSE

## Plot electric field, harmonic spectrum and $\psi$
Now we plot the electric field, harmonic spectrum and finally the wavefunction $\psi$. The parameters for the plot are set in the first cell and the following one just does the plot.

In [ ]:
# Maximal frequencty for the 
omega_max_plot = 3.5  # [a.u.]

# Optional spatial filtering of the wavefunction plot
filter_xgrid = False  # False: full x_grid; True: restrict the plotted range
x_max_plot = 250.1    # [a.u.], used only when filter_xgrid is True

# Optional lower-value filtering of the wavefunction
filter_wavefunction_min = False
wavefunction_min = 1e-8
wavefunction_max = 0.5

In [ ]:
# Load the wavefunction
t_psi, x_grid, wavefunction = output.get_wavefunction(
    inputs,
    grids=True
)


# -------------------------------------------------------------------------
# 1. Electric field
# -------------------------------------------------------------------------
fig1, ax1 = plt.subplots(figsize=(8, 5))

ax1.plot(
    output.get_tgrid(),
    output.get_Efield(),
    label='Electric field'
)

ax1.set_xlabel(r'$t~[\mathrm{a.u.}]$')
ax1.set_ylabel(r'$\mathcal{E}~[\mathrm{a.u.}]$')
ax1.legend()

fig1.tight_layout()
plt.show()


# -------------------------------------------------------------------------
# 2. Harmonic spectrum
# -------------------------------------------------------------------------
fig2, ax2 = plt.subplots(figsize=(8, 5))

ogrid = output.get_omegagrid()[:]
ko_max = mn.FindInterval(ogrid, omega_max_plot)

photon_energy = mn.ConvertPhoton(
    ogrid[:ko_max],
    'omegaau',
    'eV'
)

ax2.semilogy(
    photon_energy,
    np.abs(output.get_Fsourceterm())[:ko_max],
    label='Dipole acceleration spectrum'
)

ax2.set_xlim(photon_energy[[0, -1]])
ax2.set_xlabel(r'$\omega~[\mathrm{eV}]$')
ax2.set_ylabel(
    r'$|(\partial \hat{\jmath}/\partial t)(\omega)|'
    r'~[\mathrm{arb.~u.}]$'
)
ax2.legend()

fig2.tight_layout()
plt.show()


# -------------------------------------------------------------------------
# 3. Wavefunction
# -------------------------------------------------------------------------

# Figure size in inches: (width, height)
figsize_wavefunction = (8, 5.5)


# Select the spatial range
if filter_xgrid:
    x_range = np.abs(x_grid) < x_max_plot
else:
    x_range = slice(None)

x_plot = x_grid[x_range]


# Prepare the wavefunction data
psi_plot = np.abs(wavefunction).T[x_range]


# Optionally replace values below wavefunction_min
if filter_wavefunction_min:
    psi_display = np.maximum(
        psi_plot,
        wavefunction_min
    )
else:
    psi_display = psi_plot


# Logarithmic colour normalisation
wavefunction_norm = colors.LogNorm(
    vmin=wavefunction_min,
    vmax=wavefunction_max
)


fig3, ax3 = plt.subplots(
    figsize=figsize_wavefunction,
    layout='constrained'
)

pc3 = ax3.pcolormesh(
    t_psi,
    x_plot,
    psi_display,
    cmap='jet',
    norm=wavefunction_norm,
    shading='auto'
)

ax3.set_xlabel(r'$t~[\mathrm{a.u.}]$')
ax3.set_ylabel(r'$x~[\mathrm{a.u.}]$')

ax3.set_xlim(
    np.min(t_psi),
    np.max(t_psi)
)

ax3.set_ylim(
    np.min(x_plot),
    np.max(x_plot)
)


# Horizontal colour bar below the plot
cbar = fig3.colorbar(
    pc3,
    ax=ax3,
    orientation='horizontal',
    location='bottom',
    pad=0.08,
    shrink=0.8,
    aspect=45
)

cbar.set_label(r'$|\psi|~[\mathrm{a.u.}]$')

plt.show()

## Gabor transform

A more detailed insight comes from the Gabor transform, which resolves the generation in both time and frequency. Next cell again sets the parameters and the following one does the plot.

In [ ]:
omega_max_plot = 3.5  # [a.u.]
Tmin_Gabor = 250.0     # [a.u.]
Tmax_Gabor = 1350.0    # [a.u.]

In [ ]:
# Time-dependent source term
grad_V = output.get_sourceterm()

# Numerical parameters required by the Gabor transform
dt_Gabor = output.tgrid[1] - output.tgrid[0]
T = output.tgrid[output.Nt - 1]

# Calculate the Gabor transform
tgrid_Gabor, ogrid_Gabor, Gabor = DLL.gabor_transform(
    grad_V,
    dt_Gabor,
    output.Nt,
    omega_max_plot,
    Tmin_Gabor,
    Tmax_Gabor,
    1000,
    a=8
)


# -------------------------------------------------------------------------
# Plot the Gabor transform
# -------------------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(8, 5.5),
    layout='constrained'
)

pc = ax.pcolormesh(
    tgrid_Gabor,
    mn.ConvertPhoton(
        ogrid_Gabor,
        'omegaau',
        'eV'
    ),
    Gabor,
    cmap='jet',
    norm=colors.LogNorm(
        vmin=1e-6,
        vmax=1.0
    ),
    shading='gouraud'
)

ax.set_xlabel(r'$t~[\mathrm{a.u.}]$')
ax.set_ylabel(r'Energy $[\mathrm{eV}]$')

cbar = fig.colorbar(
    pc,
    ax=ax
)

cbar.set_label(
    r'Gabor spectrogram $[\mathrm{arb.~u.}]$'
)

plt.show()

## Test different parameters

We see a large simulation box, the exiting electrons do not contribute to HHG. We can try to run TDSE with different parameters.

Here we recreate the input dictionary and change some of the input values (we use different variables `*_test`, so we can compare the results later). We drastically reduce the number of points in $x$ and apply an absorber. this changes `num_r` input and we reduce it to 300, next we use complex absorber, which means `type`=1, and the attenuation factor together with the size of the absorber is defined.

The rest of the cell initialises the TDSE inputs and runs TDSE.

(The default values for these notebooks: no absorber `num_r`=1600, `absorber[type]`=0; with the absorber `num_r`=300, `absorber[type]`=1.)


In [ ]:
inputs_dict_test = {
    'trg_a'    : trg_a,
    'Eguess'   : Eguess,
    'dt'       : dt,
    'dx'       : dx,
    'num_r'    : 300, #16000,  # test the number of points in x
    'CV'       : CV,
    'x_int'    : x_int,
    # apply absorber
    'absorber' : {'type'  : 1,
                  'x_cap' : 50., # a.u.
                  'alpha' : 0.001},  # a.u.
    'writewft' : 1,
    'tprint'   : 1
}

# Prepare the test input structure
inputs_test = inputs_def()
# USe our modified inputs
inputs_test.init_default_inputs(**inputs_dict_test)
inputs_test.init_time_and_field(DLL, E=E, t=tgrid)
DLL.init_GS(inputs_test) # Initialise the ground state 
output_test = outputs_def() # Prepare a separate output structure
DLL.call1DTDSE(inputs_test,output_test) # Run the TDSE calculation and store the result in output_test

## Plot the results

The same block as for the plotting in the first case.

In [ ]:
omega_max_plot = 3.5  # [a.u.]

# Optional spatial filtering of the wavefunction plot
filter_xgrid = False  # False: full x_grid; True: restrict the plotted range
x_max_plot = 250.1    # [a.u.], used only when filter_xgrid is True

# Optional lower-value filtering
filter_wavefunction_min = False
wavefunction_min = 1e-8
wavefunction_max = 0.5

In [ ]:
# Load the wavefunction
t_psi, x_grid, wavefunction = output_test.get_wavefunction(
    inputs_test,
    grids=True
)


# -------------------------------------------------------------------------
# 1. Electric field
# -------------------------------------------------------------------------
fig1, ax1 = plt.subplots(figsize=(8, 5))

ax1.plot(
    output_test.get_tgrid(),
    output_test.get_Efield(),
    label='Electric field'
)

ax1.set_xlabel(r'$t~[\mathrm{a.u.}]$')
ax1.set_ylabel(r'$\mathcal{E}~[\mathrm{a.u.}]$')
ax1.legend()

fig1.tight_layout()
plt.show()


# -------------------------------------------------------------------------
# 2. Harmonic spectrum
# -------------------------------------------------------------------------
fig2, ax2 = plt.subplots(figsize=(8, 5))

ogrid = output_test.get_omegagrid()[:]
ko_max = mn.FindInterval(ogrid, omega_max_plot)

photon_energy = mn.ConvertPhoton(
    ogrid[:ko_max],
    'omegaau',
    'eV'
)

ax2.semilogy(
    photon_energy,
    np.abs(output_test.get_Fsourceterm())[:ko_max],
    label='Dipole acceleration spectrum'
)

ax2.set_xlim(photon_energy[[0, -1]])
ax2.set_xlabel(r'$\omega~[\mathrm{eV}]$')
ax2.set_ylabel(
    r'$|(\partial \hat{\jmath}/\partial t)(\omega)|'
    r'~[\mathrm{arb.~u.}]$'
)
ax2.legend()

fig2.tight_layout()
plt.show()


# -------------------------------------------------------------------------
# 3. Wavefunction
# -------------------------------------------------------------------------

# Figure size in inches: (width, height)
figsize_wavefunction = (8, 5.5)


# Select the spatial range
if filter_xgrid:
    x_range = np.abs(x_grid) < x_max_plot
else:
    x_range = slice(None)

x_plot = x_grid[x_range]


# Prepare the wavefunction data
psi_plot = np.abs(wavefunction).T[x_range]


# Optionally replace values below wavefunction_min
if filter_wavefunction_min:
    psi_display = np.maximum(
        psi_plot,
        wavefunction_min
    )
else:
    psi_display = psi_plot


# Logarithmic colour normalisation
wavefunction_norm = colors.LogNorm(
    vmin=wavefunction_min,
    vmax=wavefunction_max
)


fig3, ax3 = plt.subplots(
    figsize=figsize_wavefunction,
    layout='constrained'
)

pc3 = ax3.pcolormesh(
    t_psi,
    x_plot,
    psi_display,
    cmap='jet',
    norm=wavefunction_norm,
    shading='auto'
)

ax3.set_xlabel(r'$t~[\mathrm{a.u.}]$')
ax3.set_ylabel(r'$x~[\mathrm{a.u.}]$')

ax3.set_xlim(
    np.min(t_psi),
    np.max(t_psi)
)

ax3.set_ylim(
    np.min(x_plot),
    np.max(x_plot)
)


# Horizontal colour bar below the plot
cbar = fig3.colorbar(
    pc3,
    ax=ax3,
    orientation='horizontal',
    location='bottom',
    pad=0.08,
    shrink=0.8,
    aspect=45
)

cbar.set_label(r'$|\psi|~[\mathrm{a.u.}]$')

plt.show()

## Gabor transform

We see the wavefunction extent is drastically reduced. However, the spectrum still contains the considered harmonic with much cleaner structure. The usual explanation is that the spatial filtering absorbs long trajectories and reduces quantum-path interferences. Let us see it in more detail and perform Gabor transform. We plot both Gabor transforms to se the direct comparison.

In [ ]:
# Time-dependent source term
grad_V_test = output_test.get_sourceterm()


# Numerical parameters required by the Gabor transform
dt_Gabor_test = (
    output_test.tgrid[1]
    - output_test.tgrid[0]
)

T_test = output_test.tgrid[
    output_test.Nt - 1
]


# Calculate the Gabor transform
tgrid_Gabor_test, ogrid_Gabor_test, Gabor_test = DLL.gabor_transform(
    grad_V_test,
    dt_Gabor_test,
    output_test.Nt,
    omega_max_plot,
    Tmin_Gabor,
    Tmax_Gabor,
    1000,
    a=8
)


# -------------------------------------------------------------------------
# Convert the frequency axes to photon energy
# -------------------------------------------------------------------------

energy_Gabor = mn.ConvertPhoton(
    ogrid_Gabor,
    'omegaau',
    'eV'
)

energy_Gabor_test = mn.ConvertPhoton(
    ogrid_Gabor_test,
    'omegaau',
    'eV'
)


# -------------------------------------------------------------------------
# Plot the original and test Gabor transforms
# -------------------------------------------------------------------------

fig_Gabor, (ax_Gabor, ax_Gabor_test) = plt.subplots(
    1,
    2,
    figsize=(12, 5.5),
    sharex=True,
    sharey=True,
    layout='constrained'
)


# Shared logarithmic colour normalisation
Gabor_norm = colors.LogNorm(
    vmin=1e-6,
    vmax=1.0
)


# Original calculation
pc_Gabor = ax_Gabor.pcolormesh(
    tgrid_Gabor,
    energy_Gabor,
    Gabor,
    cmap='jet',
    norm=Gabor_norm,
    shading='gouraud'
)

ax_Gabor.set_title('Original parameters')
ax_Gabor.set_xlabel(r'$t~[\mathrm{a.u.}]$')
ax_Gabor.set_ylabel(r'Energy $[\mathrm{eV}]$')


# Test calculation
pc_Gabor_test = ax_Gabor_test.pcolormesh(
    tgrid_Gabor_test,
    energy_Gabor_test,
    Gabor_test,
    cmap='jet',
    norm=Gabor_norm,
    shading='gouraud'
)

ax_Gabor_test.set_title('Test parameters')
ax_Gabor_test.set_xlabel(r'$t~[\mathrm{a.u.}]$')


# Use the original Gabor grid for the common initial limits
ax_Gabor.set_xlim(
    np.min(tgrid_Gabor),
    np.max(tgrid_Gabor)
)

ax_Gabor.set_ylim(
    np.min(energy_Gabor),
    np.max(energy_Gabor)
)


# Shared horizontal colour bar below both subplots
cbar_Gabor = fig_Gabor.colorbar(
    pc_Gabor_test,
    ax=[ax_Gabor, ax_Gabor_test],
    orientation='horizontal',
    location='bottom',
    pad=0.08,
    shrink=0.8,
    aspect=45
)

cbar_Gabor.set_label(
    r'Gabor spectrogram $[\mathrm{arb.~u.}]$'
)

plt.show()

The Gabor transform supports the claim that the long trajectories are filtered as we see only the left sides of the 'arcs' in the test case. The maximal energy in the Gabor transform is slightly lower in the test case. It is important to mention that this is a showcase of the capabilities, the attenuated region spans 50 a.u., which means it starts 10 a.u. from the origin, affecting strongly physics in all the range. So, this notebook can serve as a starting point for more experiments.

### Thank you for reaching the end of the tutorial.